[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/trigonometry/t07_products.ipynb)

# How astronomers multiplied before logarithms: product to sum, and back

At Tycho Brahe's observatory in 1580, Paul Wittich showed a trick: a product of two cosines becomes a sum of two
cosines. Adding is easier than multiplying, so for thirty years, before logarithms, astronomers multiplied this way.
Johannes Werner had used the idea earlier, but never published it.

The picture behind it: **two arrows, one turned forward by y and one turned back by y.**

**How to use this notebook.** Press **Run all** (or click a grey box and press Shift + Enter, box by box, from the
top). Then drag the sliders and watch. Change any number you like and run the box again. Nothing can break: if
something looks odd, run everything again from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this box first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

def circle_axes(ax, lim=1.35, labels=True):
    """The unit circle with light axes: the stage for the turning arrow."""
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), color=DIM, lw=1.6)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(False)
    if labels:
        for x, y, s in ((1, 0, "1"), (-1, 0, "−1"), (0, 1, "1"), (0, -1, "−1")):
            ax.annotate(s, (x, y), xytext=(5, 5), textcoords="offset points", color=DIM, fontsize=11)
    return ax

def arrow(ax, angle, color=CORAL, lw=3.2, r=1.0, label=None, start=(0.0, 0.0)):
    """The turning arrow: length r, at `angle` radians, from `start`. Returns its tip."""
    x, y = start[0] + r * np.cos(angle), start[1] + r * np.sin(angle)
    ax.annotate("", xy=(x, y), xytext=start, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=18))
    if label: ax.annotate(label, (x, y), xytext=(8, 6), textcoords="offset points", color=color)
    return x, y

def turn_arc(ax, angle, r=0.28, color=DIM):
    """A small arc from 0 to `angle`, showing which way the arrow turned."""
    t = np.linspace(0, angle, 100)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, lw=1.4)

def pi_ticks(ax, lo=-2 * np.pi, hi=2 * np.pi, step=np.pi / 2):
    """x ticks at multiples of π/2, labelled with π."""
    names = {0: "0", 1: "π/2", 2: "π", 3: "3π/2", 4: "2π", 5: "5π/2", 6: "3π",
             -1: "−π/2", -2: "−π", -3: "−3π/2", -4: "−2π", -5: "−5π/2", -6: "−3π"}
    ks = range(int(np.ceil(lo / step)), int(np.floor(hi / step)) + 1)
    ax.set_xticks([k * step for k in ks]); ax.set_xticklabels([names.get(k, f"{k}π/2") for k in ks])

def gaps(y, big=8.0):
    """Break a curve where it shoots off to infinity, so no false vertical lines are drawn."""
    y = np.array(y, dtype=float); y[np.abs(y) > big] = np.nan
    return y

def both_sides(ax, x, left, right, title, ylim=None, big=8.0):
    """An identity as two graphs: the left side thick, the right side dashed on top. If they are the same function,
    the dashes sit exactly on the thick line."""
    ax.plot(x, gaps(left, big), color=TEAL, lw=6, alpha=0.55, label="left side")
    ax.plot(x, gaps(right, big), color=CORAL, lw=1.8, ls="--", label="right side")
    ax.set_title(title, fontsize=13)
    if ylim: ax.set_ylim(*ylim)
    pi_ticks(ax, x.min(), x.max())

def check_identity(left, right, two=False, n=6, lo=-2 * np.pi, hi=2 * np.pi, ok=None, seed=0):
    """Check an identity at random angles: prints a small table and the largest difference (relative), which should
    be about 10⁻¹⁵, far below the 10⁻¹² we ask for. `ok` keeps away from angles where a side is undefined."""
    rng = np.random.default_rng(seed)
    x, y = rng.uniform(lo, hi, 400), rng.uniform(lo, hi, 400)
    keep = ok(x, y) if (ok and two) else (ok(x) if ok else np.ones_like(x, dtype=bool))
    x, y = x[keep][:n], y[keep][:n]
    L, R = (left(x, y), right(x, y)) if two else (left(x), right(x))
    head = f"{'x':>9}" + (f"{'y':>9}" if two else "") + f"{'left side':>16}{'right side':>16}"
    print(head)
    for i in range(len(x)):
        print(f"{x[i]:9.4f}" + (f"{y[i]:9.4f}" if two else "") + f"{L[i]:16.10f}{R[i]:16.10f}")
    gap = float(np.max(np.abs(L - R) / np.maximum(1.0, np.abs(R))))
    print(f"largest difference: {gap:.1e}")
    assert gap < 1e-12, gap
    return gap

away_from_sin0 = lambda x: np.abs(np.sin(x)) > 0.05    # keeps cot and cosec away from where they break
away_from_cos0 = lambda x: np.abs(np.cos(x)) > 0.05    # keeps tan and sec away from where they break

## 1 · Halfway between two arrows

Take the arrow at x. Turn one copy forward by y, to x + y, and another back by y, to x − y. Now find the point
exactly halfway between their two tips.

That midpoint lies on the arrow at x, but shorter: its length is cos y. So its horizontal projection is cos x × cos y. But it
is also the average of the two tips' horizontal projections. That is the trick:

**2 cos x cos y = cos(x + y) + cos(x − y).**

In [ ]:
def halfway(x_deg=50.0, y_deg=35.0):
    x, y = math.radians(x_deg), math.radians(y_deg)
    fig, ax = plt.subplots(figsize=(6.8, 6.8))
    circle_axes(ax)
    a = arrow(ax, x + y, color=SAGE, lw=2.4, label="x + y")
    b = arrow(ax, x - y, color=LILAC, lw=2.4, label="x − y")
    m = ((a[0] + b[0]) / 2, (a[1] + b[1]) / 2)
    ax.plot([a[0], b[0]], [a[1], b[1]], color=DIM, ls=":")
    ax.plot([-1.3 * math.cos(x), 1.3 * math.cos(x)], [-1.3 * math.sin(x), 1.3 * math.sin(x)], color=TEAL, lw=1, ls="--")
    arrow(ax, x, r=math.cos(y), label="the midpoint")
    ax.plot([m[0], m[0]], [0, m[1]], color=CORAL, ls=":")
    ax.set_title(f"midpoint's horizontal projection = {m[0]:+.4f}\ncos x cos y = {math.cos(x) * math.cos(y):+.4f}", fontsize=12.5)
    plt.show()

interact(halfway, x_deg=FloatSlider(value=50, min=0, max=180, step=5, description="x (°)"),
         y_deg=FloatSlider(value=35, min=0, max=90, step=5, description="y (°)"));

**Notice.** However you set x and y, the midpoint stays on the line of the arrow at x. Its vertical projection gives a
second rule for free: **2 sin x cos y = sin(x + y) + sin(x − y).**

## 2 · The multiplying machine

Here is the astronomers' method for multiplying two numbers between 0 and 1:

1. Look up the angles whose cosines they are: a = cos A, b = cos B.
2. Look up cos(A − B) and cos(A + B) in the same table.
3. Add them and halve: a × b = [cos(A − B) + cos(A + B)] ÷ 2.

No multiplication of the numbers themselves, only look-ups, one addition and a halving. The machine below uses a
table in steps of one arc-minute, roughly what careful tables gave, so you can see how accurate the method was.

In [ ]:
TABLE_STEP = 1 / 60                                    # degrees: one arc-minute
def table_cos(deg):  return math.cos(math.radians(round(deg / TABLE_STEP) * TABLE_STEP))
def table_acos(v):   return round(math.degrees(math.acos(v)) / TABLE_STEP) * TABLE_STEP

def machine(a=0.7071, b=0.3420):
    A, B = table_acos(a), table_acos(b)
    p = (table_cos(A - B) + table_cos(A + B)) / 2
    print(f"1. a = {a} is cos {A:.3f}°,  b = {b} is cos {B:.3f}°   (from the table)")
    print(f"2. cos(A − B) = cos {A - B:.3f}° = {table_cos(A - B):.5f};  cos(A + B) = cos {A + B:.3f}° = {table_cos(A + B):.5f}")
    print(f"3. add and halve: {p:.5f}")
    print(f"   true product a × b = {a * b:.5f}   (error {abs(p - a * b):.1e})")

interact(machine, a=FloatSlider(value=0.7071, min=0.01, max=1.0, step=0.0001, readout_format=".4f"),
         b=FloatSlider(value=0.3420, min=0.01, max=1.0, step=0.0001, readout_format=".4f"));

**Notice.** The answer is right to about four decimal places. Big numbers work too: move the decimal point first,
just as the astronomers did (7071 × 3420 is 0.7071 × 0.3420 × 10⁸).

## 3 · Sum to product

Run the trick backwards. Put x + y = α and x − y = β, so x = (α + β) ÷ 2 and y = (α − β) ÷ 2. A sum of two waves
becomes a product:

**sin α + sin β = 2 sin((α + β)/2) cos((α − β)/2).**

## 4 · An identity you can hear: beats

Play two notes close in pitch, 440 and 442 vibrations a second. By sum to product, their sum is a note at the
average pitch, 441, multiplied by a slow cosine at half the difference. You hear it swell and fade: **beats**, at
the difference, twice a second. Piano tuners listen for exactly this when they tune two strings to one note.

In [ ]:
from IPython.display import Audio, display

def beats(f2=442.0):
    f1, rate = 440.0, 22050
    t = np.linspace(0, 0.03, 3000)
    tt = np.linspace(0, 2.0, 20000)
    fig, (ax, bx) = plt.subplots(2, 1, figsize=(11.5, 6.6))
    ax.plot(t * 1000, np.sin(2 * np.pi * f1 * t), color=TEAL, lw=1.4, label="440 Hz")
    ax.plot(t * 1000, np.sin(2 * np.pi * f2 * t), color=LILAC, lw=1.4, label=f"{f2:.0f} Hz")
    ax.set_xlabel("milliseconds"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=2); ax.set_title("two notes, a few milliseconds of each")
    s = np.sin(2 * np.pi * f1 * tt) + np.sin(2 * np.pi * f2 * tt)
    env = 2 * np.abs(np.cos(np.pi * (f2 - f1) * tt))
    bx.plot(tt, s, color=CHALK, lw=0.4, alpha=0.6)
    bx.plot(tt, env, color=CORAL, lw=2.2, label="2 cos(½ × difference): the envelope")
    bx.plot(tt, -env, color=CORAL, lw=2.2)
    bx.set_xlabel("seconds"); bx.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32))
    bx.set_title(f"their sum over 2 seconds: {abs(f2 - f1):.0f} beats a second")
    plt.tight_layout(); plt.show()
    ts = np.linspace(0, 2.0, int(rate * 2.0), endpoint=False)
    sound = 0.12 * (np.sin(2 * np.pi * f1 * ts) + np.sin(2 * np.pi * f2 * ts))      # quiet on purpose
    try:
        display(Audio(sound, rate=rate, normalize=False))
    except TypeError:                                                              # older IPython has no normalize
        display(Audio(sound, rate=rate))

interact(beats, f2=FloatSlider(value=442, min=440.5, max=450, step=0.5, description="second note (Hz)"));

**Notice.** Press play (the sound is quiet: turn your volume up slowly). Move the second note further away and the
beats get faster. Past about six a second, you stop counting them and hear roughness instead.

## 5 · The identity gallery

The class 11 list in full: product to sum (NCERT identity 21) and sum to product (NCERT identity 20). Plotted
against x with y fixed at 0.7, and checked at random x and y.

In [ ]:
x = np.linspace(-2 * np.pi, 2 * np.pi, 1500)
Y = 0.7
prod = [("2 cos x cos y = cos(x + y) + cos(x − y)", lambda x, y: 2 * np.cos(x) * np.cos(y), lambda x, y: np.cos(x + y) + np.cos(x - y)),
        ("−2 sin x sin y = cos(x + y) − cos(x − y)", lambda x, y: -2 * np.sin(x) * np.sin(y), lambda x, y: np.cos(x + y) - np.cos(x - y)),
        ("2 sin x cos y = sin(x + y) + sin(x − y)", lambda x, y: 2 * np.sin(x) * np.cos(y), lambda x, y: np.sin(x + y) + np.sin(x - y)),
        ("2 cos x sin y = sin(x + y) − sin(x − y)", lambda x, y: 2 * np.cos(x) * np.sin(y), lambda x, y: np.sin(x + y) - np.sin(x - y))]
summ = [("cos x + cos y = 2 cos((x+y)/2) cos((x−y)/2)", lambda x, y: np.cos(x) + np.cos(y), lambda x, y: 2 * np.cos((x + y) / 2) * np.cos((x - y) / 2)),
        ("cos x − cos y = −2 sin((x+y)/2) sin((x−y)/2)", lambda x, y: np.cos(x) - np.cos(y), lambda x, y: -2 * np.sin((x + y) / 2) * np.sin((x - y) / 2)),
        ("sin x + sin y = 2 sin((x+y)/2) cos((x−y)/2)", lambda x, y: np.sin(x) + np.sin(y), lambda x, y: 2 * np.sin((x + y) / 2) * np.cos((x - y) / 2)),
        ("sin x − sin y = 2 cos((x+y)/2) sin((x−y)/2)", lambda x, y: np.sin(x) - np.sin(y), lambda x, y: 2 * np.cos((x + y) / 2) * np.sin((x - y) / 2))]
fig, axs = plt.subplots(4, 2, figsize=(12, 11.5))
for ax, (name, L, R) in zip(axs.flat, prod + summ):
    both_sides(ax, x, L(x, Y), R(x, Y), name + "  (y = 0.7)", ylim=(-2.4, 2.4))
axs[0, 0].legend(loc="lower right"); plt.tight_layout(); plt.show()
for name, L, R in prod + summ:
    print(name); check_identity(L, R, two=True, n=4)

## 6 · On paper: the CBSE way

**Example 1.** Prove that (sin 5x + sin 3x) ÷ (cos 5x + cos 3x) = tan 4x.

1. Top, by sum to product: sin 5x + sin 3x = 2 sin 4x cos x.
2. Bottom: cos 5x + cos 3x = 2 cos 4x cos x.
3. Divide: the 2 cos x cancels, leaving sin 4x ÷ cos 4x = **tan 4x**.

In [ ]:
xx = np.linspace(0.05, 0.35, 7)
assert np.allclose((np.sin(5 * xx) + np.sin(3 * xx)) / (np.cos(5 * xx) + np.cos(3 * xx)), np.tan(4 * xx), atol=1e-12)
print("checked at", len(xx), "angles")

**Example 2.** Find cos 75° × cos 15°.

1. 2 cos 75° cos 15° = cos 90° + cos 60° = 0 + 1/2.
2. **cos 75° cos 15° = 1/4.**

In [ ]:
print(f"cos 75° cos 15° = {math.cos(math.radians(75)) * math.cos(math.radians(15)):.6f}")

**Try it.**
1. Find sin 75° − sin 15°.
2. Prove that (cos 7x + cos 5x) ÷ (sin 7x − sin 5x) = cot x.

<details><summary>Answers</summary>

1. 2 cos 45° sin 30° = 2 × (√2/2) × 1/2 = √2/2.
2. Top: 2 cos 6x cos x. Bottom: 2 cos 6x sin x. Divide: cos x ÷ sin x = cot x.

</details>

## Check yourself
1. Product to sum turns which operation into which?
2. Two notes at 440 and 443 Hz: how many beats a second?
3. Find 2 sin 45° cos 15°.

<details><summary>Answers</summary>

1. Multiplying into adding.
2. Three: the difference.
3. sin 60° + sin 30° = (√3 + 1) ÷ 2.

</details>

## Sources
MacTutor History of Mathematics (St Andrews): "Johannes Werner"; "Paul Wittich"; "Jost Bürgi". University of
Cambridge, Department of History and Philosophy of Science, "Tycho's mathematics". HyperPhysics (Georgia State
University), "Beats"; Simon Fraser University, Handbook for Acoustic Ecology, "Beats". NCERT class 11, chapter 3
(identities 20–21). CBSE Mathematics syllabus 2026–27. All retrieved 2026-10-21.